# Entity Vector — `entity_vectors.npy`

**Mission**: reproduce the recipe **entity (food-category) vector** of the ICOCO paper from
the new `ICOCO.duckdb` pipeline (legacy reference:
`old_code/11_entity_vector/entity_vector_npy_export.ipynb`).

**Output**: `/workspace/7_create_npy_files/entity_vectors.npy`, dtype `float32`,
shape `(50511, 28)` — one row per recipe (`silver.RecipeZenodo` rowid order), one column per
FoodUSDA *latest* food category ordered by `id` (legacy convention: `idx = id - 1`).

**Inputs**

| Input | Where |
|---|---|
| Recipe ingredient occurrences | `silver.RecipeZenodo.Ingredients` (`VARCHAR[]`) |
| Dirty ingredient → USDA category (LLM-labeled) | legacy JSON `old_code/11_entity_vector/data/dirty_ingredient_category_mapping.json` (`input`, `category`) — the new DuckDB has no copy of this table, and the mission says *do not create new tables*, so the notebook reads the file directly via `read_json` |
| The 28 fixed categories | `silver.FoodUSDA_latest_food_category` (`id`, `description`) |

**Legacy deltas (expected, documented)**: legacy shape was `(50513, 28)` (2 rows dropped in
the new bronze ingestion; alignment offsets at old rows **21475 / 23805**).

## 0. Setup — connect → close (flush WAL) → reconnect

Same connection pattern as `silver_curation.ipynb` / `explore_zenodo_recipe.ipynb`:
close first so the WAL file (`ICOCO.duckdb.wal`) is flushed, then reconnect fresh.
This notebook only **reads** DuckDB and writes **TEMP** objects (session-local, vanish on
disconnect), so re-running it never changes the persistent database state.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

In [4]:
from sql.connection import ConnectionManager

# IMPORTANT: DuckDB allows only ONE connection to the database at a time.
# Reuse the SAME connection that %sql already opened (do NOT duckdb.connect() again).
raw = ConnectionManager().current.connection_sqlalchemy.connection

# Keep RAM safe inside the 3.68 GB container: cap DuckDB memory + threads.
raw.execute("SET memory_limit = '1.5GB'")
raw.execute("SET threads = 4")
print('Reusing the single %sql connection for Python-side fetches (raw):', raw)

Reusing the single %sql connection for Python-side fetches (raw): <sqlalchemy.pool.base._ConnectionFairy object at 0x701fa96f9de0>


In [5]:
from sql.connection import ConnectionManager

# IMPORTANT: DuckDB allows only ONE connection to the database at a time.
# Reuse the SAME connection that %sql already opened (do NOT duckdb.connect() again).
raw = ConnectionManager().current.connection_sqlalchemy.connection

# Keep RAM safe inside the 3.68 GB container: cap DuckDB memory + threads.
raw.execute("SET memory_limit = '1.5GB'")
raw.execute("SET threads = 4")
print('Reusing the single %sql connection for Python-side fetches (raw):', raw)

Reusing the single %sql connection for Python-side fetches (raw): <sqlalchemy.pool.base._ConnectionFairy object at 0x701fa96f9de0>


## 1. Sanity checks

In [6]:
%%sql
SELECT 'silver.RecipeZenodo' AS tbl, count(*) AS rows FROM silver.RecipeZenodo
UNION ALL SELECT 'silver.FoodUSDA_latest_food_category', count(*) FROM silver.FoodUSDA_latest_food_category;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

tbl,rows
silver.RecipeZenodo,50511
silver.FoodUSDA_latest_food_category,28


## 2. Build the long-format vector (pure SQL)

In [7]:
%%sql
-- Long-format entity (food-category) vector entries.
-- Faithful to the legacy Python logic (old_code/11_entity_vector/entity_vector_npy_export.ipynb):
--   * ONLY categories that exist in silver.FoodUSDA_latest_food_category are counted
--     (they are filtered BEFORE the tally, so unknown categories leave BOTH the
--      numerator AND the denominator -- this matches `if cat and cat in cat_to_idx`);
--   * ingredient occurrences are counted per recipe (duplicates kept);
--   * share = category count / total valid category occurrences of the recipe.
CREATE OR REPLACE TEMP VIEW ent_long AS
WITH ing AS (
    SELECT r.rowid AS recipe_idx, lower(trim(u.ing)) AS ing
    FROM silver.RecipeZenodo r, UNNEST(r.Ingredients) AS u(ing)
),
cats AS (
    -- legacy builds its index from food_category.csv sorted by id -> idx = id - 1
    SELECT lower(trim(description)) AS cat, id - 1 AS cat_idx
    FROM silver.FoodUSDA_latest_food_category
),
matched AS (
    SELECT i.recipe_idx, c.cat_idx
    FROM ing i
    JOIN read_json(
        '/workspace/old_code/11_entity_vector/data/dirty_ingredient_category_mapping.json',
        columns = {input: 'VARCHAR', category: 'VARCHAR'}) m
      ON lower(trim(m.input)) = i.ing
    JOIN cats c ON c.cat = lower(trim(m.category))
),
tally AS (
    SELECT recipe_idx, cat_idx, COUNT(*) AS cnt,
           SUM(COUNT(*)) OVER (PARTITION BY recipe_idx) AS total
    FROM matched
    GROUP BY recipe_idx, cat_idx
)
SELECT recipe_idx, cat_idx, cnt::DOUBLE / total AS share
FROM tally;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [8]:
%%sql
SELECT * FROM ent_long LIMIT 8;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

recipe_idx,cat_idx,share
3,1,0.5714285714285714
3,0,0.14285714285714285
3,3,0.14285714285714285
3,6,0.14285714285714285
6,6,0.14285714285714285
6,10,0.7142857142857143
6,1,0.14285714285714285
38,8,0.5


## 3. Coverage statistics

In [9]:
%%sql
SELECT count(*)                  AS long_rows,
       count(DISTINCT recipe_idx) AS recipes_with_entity_signal,
       count(DISTINCT cat_idx)    AS categories_actually_used
FROM ent_long;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

long_rows,recipes_with_entity_signal,categories_actually_used
260021,50460,25


## 4. Pivot to a dense matrix and save `.npy`

In [10]:
import numpy as np
import os

OUT = '/workspace/7_create_npy_files/entity_vectors.npy'

df = raw.execute('SELECT recipe_idx, cat_idx, share FROM ent_long').df()
n_recipes    = raw.execute('SELECT count(*) FROM silver.RecipeZenodo').fetchone()[0]
n_categories = raw.execute('SELECT count(*) FROM silver.FoodUSDA_latest_food_category').fetchone()[0]

ent = np.zeros((n_recipes, n_categories), dtype=np.float32)
ent[df['recipe_idx'].to_numpy(np.int64), df['cat_idx'].to_numpy(np.int64)] = df['share'].to_numpy(np.float32)

np.save(OUT, ent)
print('saved:', OUT, '| shape:', ent.shape, '| dtype:', ent.dtype,
      '| size:', round(os.path.getsize(OUT) / 1e6, 2), 'MB')

saved: /workspace/7_create_npy_files/entity_vectors.npy | shape: (50511, 28) | dtype: float32 | size: 5.66 MB


## 5. Verification

**(a) internal** — rows with signal must sum to 1 (denominator = valid category
occurrences only), plus a spot-check.

**(b) vs the legacy artifact** — full row-level comparison under the documented 2-row drop
alignment.

In [11]:
import numpy as np

# ---- (a) internal ----
row_sums = ent.sum(axis=1)
print('rows with signal (sum > 0):', int((row_sums > 0).sum()))
print('max row sum (must be <= 1 + 1e-5):', round(float(row_sums.max()), 6))

name, ings = raw.execute(
    "SELECT Name, Ingredients FROM silver.RecipeZenodo WHERE rowid = 0").fetchone()
r0 = ent[0]
print(f'spot-check rowid 0 {name!r}')
print('  nonzero category shares:', {int(i): round(float(r0[i]), 4) for i in np.nonzero(r0)[0]})

rows with signal (sum > 0): 50460
max row sum (must be <= 1 + 1e-5): 1.0
spot-check rowid 0 'Pineapple Glaze for Ham'
  nonzero category shares: {8: 0.6667, 18: 0.3333}


In [12]:
import numpy as np

# ---- (b) vs legacy artifact (read-only, mmap) ----
OLD = '/workspace/old_code/11_entity_vector/data/entity_vectors.npy'
old = np.load(OLD, mmap_mode='r')
print('legacy shape:', old.shape, '| new shape:', ent.shape)

old_idx = np.empty(ent.shape[0], dtype=np.int64)
_o = 0
for _j in range(ent.shape[0]):
    while _o in (21475, 23805):   # old rows dropped by the new bronze ingestion
        _o += 1
    old_idx[_j] = _o
    _o += 1
old_rows = np.asarray(old[old_idx], dtype=np.float32)

row_max = np.abs(old_rows - ent).max(axis=1)
same = row_max <= 1e-6
print('identical rows:', int(same.sum()), '/', ent.shape[0],
      f'({same.mean() * 100:.3f} %)')
mism = np.nonzero(~same)[0]
print('differing new-row indices:', mism.tolist())
print('known cause: ingredient-string level differences between the legacy silver CSV')
print('and the new bronze ingestion (e.g. "peach  halved" with a double space), which')
print('change how many ingredient occurrences match the LLM category mapping.')

legacy shape: (50513, 28) | new shape: (50511, 28)
identical rows: 50504 / 50511 (99.986 %)
differing new-row indices: [5688, 28200, 36436, 36710, 42430, 45302, 49156]
known cause: ingredient-string level differences between the legacy silver CSV
and the new bronze ingestion (e.g. "peach  halved" with a double space), which
change how many ingredient occurrences match the LLM category mapping.


## 6. Disconnect

Close the jupysql connection; the `TEMP` views die with the session, leaving the
database file untouched.

In [13]:
%sql --close duckdb:////workspace/ICOCO.duckdb